## This is the testing code

In [3]:
pip install earthengine-api

  Using cached earthengine_api-0.1.401-py3-none-any.whl (349 kB)
  Using cached google_cloud_storage-2.16.0-py2.py3-none-any.whl (125 kB)
  Using cached google_auth-2.29.0-py2.py3-none-any.whl (189 kB)
  Using cached google_api_python_client-2.127.0-py2.py3-none-any.whl (12.7 MB)
  Using cached google_auth_httplib2-0.2.0-py2.py3-none-any.whl (9.3 kB)
  Using cached httplib2-0.22.0-py3-none-any.whl (96 kB)
  Using cached google_api_core-2.19.0-py3-none-any.whl (139 kB)
  Using cached uritemplate-4.1.1-py2.py3-none-any.whl (10 kB)
  Using cached proto_plus-1.23.0-py3-none-any.whl (48 kB)
  Using cached googleapis_common_protos-1.63.0-py2.py3-none-any.whl (229 kB)
  Using cached pyasn1_modules-0.4.0-py3-none-any.whl (181 kB)
  Using cached cachetools-5.3.3-py3-none-any.whl (9.3 kB)
  Using cached rsa-4.9-py3-none-any.whl (34 kB)
  Using cached pyasn1-0.6.0-py2.py3-none-any.whl (85 kB)
  Using cached google_resumable_media-2.7.0-py2.py3-none-any.whl (80 kB)
  Using cached google_cloud_core

In [1]:
import ee

## Authenticate to GEE

In [2]:
ee.Authenticate()

True

In [3]:
try:
    # Initialize Earth Engine
    ee.Initialize()
    print("Earth Engine is initialized.")
except ee.EEException as e:
    #print("The Earth Engine API failed to initialize:", e)
      logger.error('Failed : %s', repr(e))

Earth Engine is initialized.


# Testing Code, can be deleted

In [22]:
print(ee.String('Hello from the Earth Engine servers!').getInfo())
# Test with a simple Earth Engine operation
image = ee.Image('srtm90_v4')
print(image.getInfo())

Hello from the Earth Engine servers!
{'type': 'Image', 'bands': [{'id': 'elevation', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': -32768, 'max': 32767}, 'dimensions': [432000, 144000], 'crs': 'EPSG:4326', 'crs_transform': [0.000833333333333, 0, -180, 0, -0.000833333333333, 60]}], 'version': 1494271934303000.0, 'id': 'srtm90_v4', 'properties': {'system:time_start': 950227200000, 'system:time_end': 951177600000, 'system:asset_size': 18827626666}}


# Project Example, should be replaced with real code

### Download the satellite image

In [16]:

# Define the area of interest with coordinates.
coordinates = [
    [-122.4481, 37.7599],
    [-122.5012, 37.7324],
    [-122.5391, 37.7071],
    [-122.4904, 37.6887],
    [-122.4376, 37.7228]
]
aoi = ee.Geometry.Polygon(coordinates)

# Filter Sentinel-2 Surface Reflectance image collection for given date range and area.
collection = (ee.ImageCollection('COPERNICUS/S2_SR')
              .filterDate('2020-01-01', '2020-01-10')
              .filterBounds(aoi)
              .first())



In [19]:
# Load a Sentinel-2 image
image = ee.ImageCollection('COPERNICUS/S2_SR')\
            .filterBounds(aoi)\
            .filterDate('2020-01-01', '2020-01-10')\
            .first()

# Compute NDVI
ndvi = image.normalizedDifference(['B8A', 'B4']).rename('NDVI')

# Define visualization parameters
vis_params = {
    'min': 0,
    'max': 1,
    'palette': ['blue', 'white', 'green']
}

# Create a visualized image
visualized_ndvi = ndvi.visualize(**vis_params)


### Option One: Download image manually

In [ ]:
# Generate a download URL
url = visualized_ndvi.getDownloadURL({
    'scale': 30,
    'crs': 'EPSG:4326',
    'region': aoi.toGeoJSONString(),
    'format': 'png'
})

print("Download URL:", url)

### Option Two: Save in Google Drive

In [21]:
# Set up export task
task = ee.batch.Export.image.toDrive(
    image=visualized_ndvi,
    description='NDVI_to_Drive',
    folder='EarthEngineImages',
    fileNamePrefix='NDVI_Image',
    scale=30,
    region=aoi,
    fileFormat='GeoTIFF'
)

# Start the task
task.start()

print("Export Task started, check Google Drive for completion.")

Export Task started, check Google Drive for completion.


## ToDo Image training and then uploading

In [10]:
# from google.cloud import storage